# Lab 3: Clustering (k-means) and Classification (Neural Networks)

## Learning Objectives

By the end of this lab, you will be able to:
- **Describe** how k-means clustering works
- **Apply** k-means to real-world data (MNIST handwritten digits)
- **Evaluate** clustering and recognise the challenges in evaluation 
- **Build and train** a neural network classifier using TensorFlow/Keras
- **Perform** hyperparameter tuning and analyze its impact on model performance

---

## Table of Contents

### Part A: k-means Clustering 

-  **Task 1: Exploring k-means Failures - k-means parameters**
-  **Task 2: Exploring k-means Failures - cluster shape**
-  Clustering Evaluation Metrics

**A.2: Applying Clustering to MNIST**
- Step 1: Load the Data
- Step 2: Explore and visualize the data
- Step 3: Apply k-means (k=10)
    - **Task 3: Complete code**
- Step 4: Evaluate clustering results
    - **Task 4: Answer five questions**
- Step 5: Finding the Right k
    - **Task 5: Complete code**
    - **Task 6: Evaluate clustering performance on task.**

### Part B: Classification with Neural Networks

- Step 1: Prepare the Data
- Step 2: Build the Model
- Step 3: Train the Model
- Step 4: Evaluate on Test Set
- Step 5: Evaluate Training Curves
- Step 6: Confusion Matrix
    - **Task 7: Evaluate the model**
- **Task 8: Hyperparameter Search**

---

# Part A: k-means clustering




**How it works:**
- The k-means algorithm partitions data points into **k clusters** by assigning each instance to the cluster with the nearest mean (centroid)
- This is an **iterative algorithm** that typically converges quickly
- A critical challenge is selecting the appropriate value of **k**, as this must be specified before running the algorithm
- The final result is **highly dependent on the initial centroid positions**

### Task 1: Exploring k-means Failures - k-means parameters

- Run the following three code cells to launch the interactive visualization.
- Play with the sliders to see how the clustering results change.
- Set **True Blobs = 4**,  **K (clusters) = 4** and **Spread = 1**. Find a combination of the rest of the parameters for which the predicted clusters (after convegence) **do not agree** with the real clusters.
  - **Include the image in your report**.
- **Explain** in a few sentences in your report why k-means fails in this case.



In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, Checkbox
from IPython.display import clear_output

import tensorflow as tf

from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs, make_moons, load_digits
from sklearn.model_selection import train_test_split

from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, silhouette_samples, \
                            accuracy_score, classification_report, silhouette_score, calinski_harabasz_score


In [ ]:
COLORS = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#EE3377', '#BBBBBB']  # Tol's palette
MARKERS = ['o', 's', '^', 'D', 'v', 'P']  # circle, square, triangle, diamond, etc.

def kmeans_step_by_step(n_blobs=3, spread=1.0, k=3, init_seed=1, step=0, show_arrows=True):
    clear_output(wait=True)
    X, y_true = make_blobs(n_samples=200, centers=n_blobs, cluster_std=spread, random_state=42)
    
    rng = np.random.RandomState(init_seed)
    centroid_indices = rng.choice(len(X), k, replace=False)
    centroids = X[centroid_indices].copy()
    
    centroid_history = [centroids.copy()]
    labels_history = [None]
    
    for i in range(15):
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)
        labels_history.append(labels.copy())
        
        new_centroids = np.array([X[labels == j].mean(axis=0) if np.sum(labels == j) > 0 
                                   else centroids[j] for j in range(k)])
        
        if np.allclose(centroids, new_centroids):
            centroid_history.append(new_centroids.copy())
            break
            
        centroids = new_centroids
        centroid_history.append(centroids.copy())
    
    max_step = len(centroid_history) - 1
    step = min(step, max_step)
    
    final_labels = labels_history[-1]
    final_centroids = centroid_history[-1]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Left plot: True clusters with shapes
    for j in range(n_blobs):
        mask = y_true == j
        axes[0].scatter(X[mask, 0], X[mask, 1], c=COLORS[j % len(COLORS)], 
                       marker=MARKERS[j % len(MARKERS)], s=50, alpha=0.7, 
                       label=f'Blob {j+1}', edgecolors='white', linewidths=0.5)
    axes[0].set_title(f'True Data: {n_blobs} blobs, spread={spread}', fontsize=11)
    axes[0].set_xlabel('Feature 1')
    axes[0].set_ylabel('Feature 2')
    axes[0].legend(loc='upper right', fontsize=8)
    
    # Right plot: K-means progress with shapes
    if step == 0:
        axes[1].scatter(X[:, 0], X[:, 1], c='gray', marker='o', s=40, alpha=0.5)
        for j in range(k):
            axes[1].scatter(centroid_history[0][j, 0], centroid_history[0][j, 1], 
                           c=COLORS[j % len(COLORS)], marker=MARKERS[j % len(MARKERS)], 
                           s=400, edgecolors='black', linewidths=2, zorder=5)
        axes[1].set_title(f'Step 0: Random Init (seed={init_seed})', fontsize=11)
    else:
        labels = labels_history[step]
        for j in range(k):
            mask = labels == j
            if mask.sum() > 0:
                axes[1].scatter(X[mask, 0], X[mask, 1], c=COLORS[j % len(COLORS)], 
                               marker=MARKERS[j % len(MARKERS)], s=50, alpha=0.7,
                               edgecolors='white', linewidths=0.5)
        
        for j in range(k):
            axes[1].scatter(centroid_history[step][j, 0], centroid_history[step][j, 1], 
                           c=COLORS[j % len(COLORS)], marker=MARKERS[j % len(MARKERS)], 
                           s=400, edgecolors='black', linewidths=2, zorder=5)
        
        if show_arrows and step > 0:
            for j in range(k):
                prev = centroid_history[step-1][j]
                curr = centroid_history[step][j]
                if not np.allclose(prev, curr):
                    axes[1].annotate('', xy=curr, xytext=prev,
                                    arrowprops=dict(arrowstyle='->', color='black', lw=2))
        
        if step == max_step:
            axes[1].set_title(f'Step {step}: Converged!', fontsize=11)
        else:
            axes[1].set_title(f'Step {step}: Assign → Update', fontsize=11)
    
    axes[1].set_xlabel('Feature 1')
    axes[1].set_ylabel('Feature 2')
    
    if step == 0:
        msg = f"Random init with seed={init_seed}\n"
    elif step == max_step:
        msg = f"Converged!"
    else:
        msg = "Assigning points → Moving centroids"
    
    axes[1].text(0.02, 0.98, msg, transform=axes[1].transAxes, fontsize=9,
                 verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.show()


**Interactive visualization description**:

The figure below has two subplots. In both plots, each cluster has a distinct color and marker.

**Left subplot — True Data:**
- Shows generated data with their ground-truth clusters
- `True Blobs`: number of clusters in the generated data
- `Spread`: how spread out each cluster is around its center

**Right subplot — k-means Iterative Clustering:**
- `K (clusters)`: number of clusters the algorithm will try to find
- `Init Seed`: controls the random initial centroid positions (visible when Iteration = 0)
- `Iteration`: step through the k-means algorithm iteration by iteration
- Lines connect each centroid to its previous position

In [ ]:
plt.style.use('default')
interact(kmeans_step_by_step,
         n_blobs=IntSlider(min=2, max=6, value=3, description='True Blobs'),
         spread=FloatSlider(min=0.3, max=2.5, step=0.1, value=1.0, description='Spread'),
         k=IntSlider(min=2, max=6, value=3, description='K (clusters)'),
         init_seed=IntSlider(min=1, max=20, value=1, description='Init Seed'),
         step=IntSlider(min=0, max=15, value=0, description='Iteration'),
         show_arrows=Checkbox(value=True, description='Show arrows'))

### Task 2: Exploring k-means failures - cluster shape
- Explore how k-means performs on the **moons dataset** below
- Can the algorithm identify correctly the two clusters? Why? **Explain in your report.** 

In [ ]:
%matplotlib inline
plt.style.use('default')


COLORS = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#EE3377', '#BBBBBB']
MARKERS = ['o', 's', '^', 'D', 'v', 'P']

def kmeans_moons(noise=0.05, k=2, init_seed=1, step=0):
    X, y_true = make_moons(n_samples=300, noise=noise, random_state=42)
    
    rng = np.random.RandomState(init_seed)
    centroid_indices = rng.choice(len(X), k, replace=False)
    centroids = X[centroid_indices].copy()
    
    centroid_history = [centroids.copy()]
    labels_history = [None]
    
    for i in range(15):
        distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
        labels = np.argmin(distances, axis=1)
        labels_history.append(labels.copy())
        
        new_centroids = np.array([X[labels == j].mean(axis=0) if np.sum(labels == j) > 0 
                                   else centroids[j] for j in range(k)])
        
        if np.allclose(centroids, new_centroids):
            centroid_history.append(new_centroids.copy())
            break
            
        centroids = new_centroids
        centroid_history.append(centroids.copy())
    
    max_step = len(centroid_history) - 1
    step = min(step, max_step)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Left: True clusters
    for j in range(2):
        mask = y_true == j
        axes[0].scatter(X[mask, 0], X[mask, 1], c=COLORS[j], marker=MARKERS[j], 
                       s=50, alpha=0.7, edgecolors='white', linewidths=0.5)
    axes[0].set_title('True Clusters (2 moons)', fontsize=11)
    axes[0].set_xlabel('Feature 1')
    axes[0].set_ylabel('Feature 2')
    
    # Right: K-means result
    if step == 0:
        axes[1].scatter(X[:, 0], X[:, 1], c='gray', s=40, alpha=0.5)
        for j in range(k):
            axes[1].scatter(centroid_history[0][j, 0], centroid_history[0][j, 1], 
                           c=COLORS[j], marker=MARKERS[j], s=400, 
                           edgecolors='black', linewidths=2, zorder=5)
        axes[1].set_title(f'Step 0: Random Init (seed={init_seed})', fontsize=11)
    else:
        labels = labels_history[step]
        for j in range(k):
            mask = labels == j
            if mask.sum() > 0:
                axes[1].scatter(X[mask, 0], X[mask, 1], c=COLORS[j], 
                               marker=MARKERS[j], s=50, alpha=0.7,
                               edgecolors='white', linewidths=0.5)
        
        for j in range(k):
            axes[1].scatter(centroid_history[step][j, 0], centroid_history[step][j, 1], 
                           c=COLORS[j], marker=MARKERS[j], s=400, 
                           edgecolors='black', linewidths=2, zorder=5)
        
        if step == max_step:
            axes[1].set_title(f'Step {step}: Converged!', fontsize=11)
        else:
            axes[1].set_title(f'Step {step}: Assign → Update', fontsize=11)
    
    axes[1].set_xlabel('Feature 1')
    axes[1].set_ylabel('Feature 2')
    
    plt.tight_layout()
    plt.show()

interact(kmeans_moons,
         noise=FloatSlider(min=0.01, max=0.3, step=0.01, value=0.05, description='Noise'),
         k=IntSlider(min=2, max=5, value=2, description='K'),
         init_seed=IntSlider(min=1, max=20, value=1, description='Init Seed'),
         step=IntSlider(min=0, max=15, value=0, description='Iteration'))

**Extra material:**
- You can explore other clustering algorithms and how they perform on various cluster shapes on the [scikit-learn clustering page](https://scikit-learn.org/stable/modules/clustering.html).

---

## A.2 — Applying Clustering to MNIST

In this section you will:
- **Explore** the MNIST handwritten digits dataset
- **Apply** k-means clustering to the dataset
- **Evaluate** k-means performance using both ground truth labels and unsupervised metrics
- **Investigate** whether k-means can identify the true number of digit classes without knowing the labels

**Load the Data**

We use scikit-learn's built-in `load_digits` dataset — a collection of 1,797 images of handwritten digits (0–9), each 8x8 pixels.

In [ ]:
digits = load_digits()

**Visualize and Explore the Data**

Let's look at some random samples from the dataset to understand what we're working with.

In [ ]:
print(digits.data.shape)
plt.style.use('dark_background')
rng = np.random.default_rng(seed=None)
indices = rng.choice(len(digits.images), size=16, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for ax, idx in zip(axes.flat, indices):
    ax.matshow(digits.images[idx], cmap="gray")
    ax.set_title(f"Label: {digits.target[idx]}")
    ax.axis("off")

plt.tight_layout()
plt.show()

**t-SNE Visualization**

- A useful visualization tool that helps us understand the structure of our data is t-SNE (t-distributed Stochastic Neighbor Embedding). This method is a nonlinear dimensionality reduction technique for visualizing high-dimensional data in 2D.
- It preserves **local neighborhood structure**, often revealing clusters not visible in linear projections.
- Note: t-SNE is for **visualization only** — it is not suitable as a preprocessing step for clustering. - Learn more about the method here:
    - https://distill.pub/2016/misread-tsne/
    - https://www.youtube.com/watch?v=NEaUSP4YerM

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, random_state=42)
X_tsne = tsne.fit_transform(digits.data)

fig, ax = plt.subplots(figsize=(10, 7))

scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=digits.target, cmap='tab10', s=20, alpha=0.7)
cbar = plt.colorbar(scatter, ax=ax, ticks=range(10))
cbar.set_label('Digit')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.set_title('t-SNE Visualization of MNIST Digits', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

Let's apply k-means with k=10.

We know there are 10 digit classes (0–9). Let's run k-means with k=10 and evaluate the clustering result.

In [ ]:
X = digits.data
y_true = digits.target

# Run k-means with k=10
kmeans_10 = KMeans(n_clusters=10, n_init=10, random_state=42)

# Prediction
labels_10 = kmeans_10.fit_predict(X)

### Task 3: Evaluate the result

Based on the **confusion matrix** in the following cell, answer the following five questions for both clusters and digits:

1. Which clusters are homogeneous (contain mostly one digit)? Identify the top 2 most homogeneous clusters.

2. Which clusters are mixed (contain many different digits)? Identify the top 2 most mixed clusters.

3. Which digits are clearly identified (most of their samples fall into a single cluster)? Choose the top 2 digits.

4. Which digits are not clearly identified (their samples are spread across many clusters)? Identify at least 2 digits.

5. Which digits are confused with each other? Identify at least two pairs of digits that are often confused.

**Justify your answers using the counts in the confusion matrix** (dominant entries vs. spread across rows/columns), and **include the confusion matrix in your report**.

In [ ]:
cm = confusion_matrix(y_true, labels_10)
df_cm = pd.DataFrame(cm,
                     index=[f"Digit {i}" for i in range(10)],
                     columns=[f"Cluster {i}" for i in range(cm.shape[1])])
print(df_cm.to_string())

**Clustering Evaluation Metrics**

In the previous examples, we created visually clear clusters with known ground truths and inspected visually how well clustering algorithms perform. 

When true labels are **unknown**, we need metrics that evaluate cluster quality based only on the data itself. Two metrics used are:

Here we will use  the **silhouette coefficient**, **silhouette coefficient diagrams**  and the **Calinski–Harabasz index (CHI)**.

Read more about these metrics: 
- https://scikit-learn.org/stable/modules/clustering.html#clustering-evaluation
- [scikit-learn Silhouette Analysis](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html)
- Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow, Chapter 9

Let's evaluate these metrics for our case.

In [ ]:
# Evaluate with unsupervised metrics
sil = silhouette_score(X, labels_10)
chi = calinski_harabasz_score(X, labels_10)

print(f"K-Means with k=10 (true number of digits)")
print(f"  Silhouette Score:            {sil:.3f}")
print(f"  Calinski-Harabasz Index:     {chi:.1f}")

### Task 4: Can We Find the Right k Without Knowing the True Labels? 
- 1. Run k-means on the MNIST dataset for k values between 2 and 21. 
- 2. Evaluate Silhouette coefficient and Calinski–Harabasz index (CHI) for each k.
- 3. Plot the values of the metrics against k.
- 4. Identify the best k for each metric, according to the max value.
- 5. Does the k agree with the known number of clusters? 
- 6. Can k-means identify correctly the number of different digits? How do the different metrics perform? Put your answer in the report and justify it. Include the plots for each metric in your report.

In [ ]:
#***********************************
# YOUR CODE GOES HERE:



#***********************************   


**Silhouette Diagrams**

An additional tool for evaluating clustering is the Silhouette Diagrams.

- Silhouette diagrams show the silhouette coefficient for **each individual sample**, grouped by cluster
- They help visualize cluster quality beyond a single average score — revealing whether **all clusters** are well-formed or if some are poorly separated
- Below we plot diagrams for k-1, k, and k+1 around the best silhouette score

**If you want you can read more:**:
- [scikit-learn Silhouette Analysis](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html)
- Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow, Chapter 9

Below is an example of the diagrams for different k values.  

In [ ]:
plt.style.use('default')
k_values = [9, 10, 11]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, k in zip(axes, k_values):
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    cluster_labels = km.fit_predict(X)
    
    sil_avg = silhouette_score(X, cluster_labels)
    sample_silhouette_values = silhouette_samples(X, cluster_labels)
    
    y_lower = 10
    for i in range(k):
        cluster_values = sample_silhouette_values[cluster_labels == i]
        cluster_values.sort()
        
        y_upper = y_lower + len(cluster_values)
        color = plt.cm.tab10(i / k)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_values,
                         facecolor=color, edgecolor=color, alpha=0.7)
        ax.text(-0.05, y_lower + 0.5 * len(cluster_values), str(i), fontsize=9, fontweight='bold')
        y_lower = y_upper + 10
    
    ax.axvline(x=sil_avg, color='red', linestyle='--', linewidth=1.5, label=f'Average: {sil_avg:.3f}')
    ax.set_title(f'k = {k}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Silhouette Coefficient')
    ax.set_ylabel('Cluster')
    ax.set_yticks([])
    ax.legend(loc='upper right', fontsize=9)

plt.suptitle('Silhouette Diagrams', fontsize=14, fontweight='bold') 
plt.tight_layout()
plt.show()

---

# Part B: Neural Networks

In this part we will build a neural network classifier for the MNIST digits dataset using TensorFlow/Keras. We will:

- **Prepare** the data (train/validation/test split, normalization, one-hot encoding)
- **Build** a fully-connected neural network
- **Train** the model and evaluate performance
- **Visualize** training curves (loss and accuracy)
- **Perform** hyperparameter tuning to improve the model

**Prepare the Data**

- Normalize pixel values to [0, 1]
- One-hot encode the labels (10 classes: digits 0–9)
- Split data into **training** (20%), **validation** (40%), and **test** (40%) sets

In [ ]:
print("TensorFlow version:", tf.__version__)

# Use the same digits dataset from Part A
X = digits.data / 16.0  # Normalize pixel values to [0, 1] (max pixel value in sklearn digits is 16)
y = digits.target

# One-hot encode the labels (10 classes: digits 0-9)
y_onehot = tf.keras.utils.to_categorical(y, num_classes=10)

# Split into train, validation and test
X_train, X_, y_train, y_ = train_test_split(X, y_onehot, test_size=0.8, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_, y_, test_size=0.5, random_state=42) 

print(f"Training set:   {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")
print(f"Test set:       {X_test.shape[0]} samples")
print(f"Input shape:    {X_train.shape[1]} features (8x8 pixels)")
print(f"Output shape:   {y_train.shape[1]} classes")

**Let's build the Neural Network Model**

- A simple **fully-connected (Dense) network** with:
  - Input layer: 64 features (8x8 pixels flattened)
  - Hidden layers with ReLU activation
  - Output layer: 10 neurons with softmax activation (one per digit class)

In [ ]:
# model architecture 
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(2, activation='relu', input_shape=(64,)),
    tf.keras.layers.Dense(6, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

In [ ]:
# compilation
learning_rate = 0.001
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# model summary
model.summary()

**Let's train the Model**

- We train using the **Adam optimizer** and **categorical cross-entropy** loss
- Training runs for 50 epochs with batch size 32
- Validation data is used to monitor performance during training

In [ ]:
number_of_epochs = 300
batch_size = 64

history = model.fit(X_train, y_train,
                    batch_size=batch_size,
                    epochs=number_of_epochs,
                    validation_data=(X_val, y_val),
                    verbose=1)

**Evaluate on Test Set**

In [ ]:
train_loss, train_accuracy = model.evaluate(X_train, y_train)
val_loss, val_accuracy = model.evaluate(X_val, y_val)
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print ('-'*32)
print(f"Train loss    : {train_loss:.4f} | Train accuracy : {train_accuracy:.4f}")
print(f"Val   loss    : {val_loss:.4f} | Val accuracy   : {val_accuracy:.4f}")
print(f"Test loss     : {test_loss:.4f} | Test accuracy  : {test_accuracy:.4f}")
print ('-'*32)


**Plot Training Curves**

- **Training curves** show how loss and accuracy evolve over epochs
- Comparing **training vs. validation** curves helps detect:
  - **Overfitting**: training loss decreases but validation loss increases
  - **Underfitting**: both losses remain high
  - **Good fit**: both losses decrease and converge

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Evaluate on test set for reference line
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)

# Loss curves
axes[0].plot(range(number_of_epochs), history.history['loss'], 'o-', color='#CC3311', markersize=3, label='Training loss')
axes[0].plot(range(number_of_epochs), history.history['val_loss'], 'o-', color='#0077BB', markersize=3, label='Validation loss')
axes[0].axhline(y=test_loss, color='#009988', linestyle=':', linewidth=2, label=f'Test loss ({test_loss:.4f})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss over Epochs')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(range(number_of_epochs), history.history['accuracy'], 'o-', color='#CC3311', markersize=3, label='Training accuracy')
axes[1].plot(range(number_of_epochs), history.history['val_accuracy'], 'o-', color='#0077BB', markersize=3, label='Validation accuracy')
axes[1].axhline(y=test_accuracy, color='#009988', linestyle=':', linewidth=2, label=f'Test accuracy ({test_accuracy:.4f})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy over Epochs')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Confusion Matrix**

In [ ]:
y_pred = np.argmax(model.predict(X_test), axis=1)
y_true_labels = np.argmax(y_test, axis=1)

cm = confusion_matrix(y_true_labels, y_pred)

fig, ax = plt.subplots(figsize=(8, 7))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(10))
disp.plot(ax=ax, cmap='Blues', colorbar=True)
ax.set_title('Confusion Matrix on Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Task 5: Evaluate the model

Answer the following questions in your report.

1. Based on the loss training curve, how does the model perform?  
2. Based on the test confusion matrix and the classification report, identify top 2 digits that are correctly classified, and 2 digits that the model confuses.

**Read More:** Diagnosing model performance: https://machinelearningmastery.com/learning-curves-for-diagnosing-machine-learning-model-performance/


### Task 6: Change train/val/test split

The baseline model above performs poorly. The model currently trains on too few data points. 
Change the train/validation/test split to 80/10/10. How does this affect the model performance? Include in your report a short discussion on the effect of the change. 

In [ ]:
#***********************************
# YOUR CODE GOES HERE:



#***********************************   

### Task 7: Hyperparameter Optimization

**Systematically explore different hyperparameter configurations** to reduce the test loss. You should train at least **9 different models** by varying at leas 2 of the parameters below. The train/val/test split should be 80/10/10.

Experiment with some of the following:

- Change the optimizer's learning rate and try values such as 0.1, 0.01, 0.001, and 0.0001.
- Change the number of hidden layers (e.g., 1, 2, or 3) and the number of neurons per layer (e.g., 16, 32, 64, 128). The baseline uses only 2 and 6 neurons — can you do better?
- Vary the batch size (e.g., 16, 32, 64). Observe how these affect convergence speed and overfitting.


**In your report, include:**
1. A table with at least 9 configurations, their validation and test losses and accuracies.
2. Loss training curve for your best model.

**Make sure** the jupyter notebook that you will attach with the report includes your best model.

In [ ]:
#***********************************
# YOUR CODE GOES HERE:



#***********************************   

## List of tasks:

## Part A: k-means Clustering 
### Task 1: Exploring k-means Failures - k-means parameters

- Run the following three code cells to launch the interactive visualization.
- Play with the sliders to see how the clustering results change.
- Set **True Blobs = 4**,  **K (clusters) = 4** and **Spread = 1**. Find a combination of the rest of the parameters for which the predicted clusters (after convegence) **do not agree** with the real clusters.
  - **Include the image in your report**.
- **Explain** in a few sentences in your report why k-means fails in this case.

### Task 2: Exploring k-means failures - cluster shape
- Explore how k-means performs on the **moons dataset** below
- Can the algorithm identify correctly the two clusters? Why? **Explain in your report.** 


### Task 3: Evaluate the result

Based on the **confusion matrix** in the following cell, answer the following five questions for both clusters and digits:

1. Which clusters are homogeneous (contain mostly one digit)? Identify the top 2 most homogeneous clusters.

2. Which clusters are mixed (contain many different digits)? Identify the top 2 most mixed clusters.

3. Which digits are clearly identified (most of their samples fall into a single cluster)? Choose the top 2 digits.

4. Which digits are not clearly identified (their samples are spread across many clusters)? Identify at least 2 digits.

5. Which digits are confused with each other? Identify at least two pairs of digits that are often confused.

**Justify your answers using the counts in the confusion matrix** (dominant entries vs. spread across rows/columns), and **include the confusion matrix in your report**.

### Task 4: Can We Find the Right k Without Knowing the True Labels? 
- 1. Run k-means on the MNIST dataset for k values between 2 and 21. 
- 2. Evaluate Silhouette coefficient and Calinski–Harabasz index (CHI) for each k.
- 3. Plot the values of the metrics against k.
- 4. Identify the best k for each metric, according to the max value.
- 5. Does the k agree with the known number of clusters? 
- 6. Can k-means identify correctly the number of different digits? How do the different metrics perform? Put your answer in the report and justify it. Include the plots for each metric in your report.

## Part B: Classification with Neural Networks
### Task 5: Evaluate the model

Answer the following questions in your report.

1. Based on the loss training curve, how does the model perform?  
2. Based on the test confusion matrix and the classification report, identify top 2 digits that are correctly classified, and 2 digits that the model confuses.

**Read More:** Diagnosing model performance: https://machinelearningmastery.com/learning-curves-for-diagnosing-machine-learning-model-performance/


### Task 6: Change train/val/test split

The baseline model above performs poorly. The model currently trains on too few data points. 
Change the train/validation/test split to 80/10/10. How does this affect the model performance? Include in your report a short discussion on the effect of the change. 

### Task 7: Hyperparameter Optimization

**Systematically explore different hyperparameter configurations** to reduce the test loss. You should train at least **9 different models** by varying at leas 2 of the parameters below. The train/val/test split should be 80/10/10.

Experiment with some of the following:

- Change the optimizer's learning rate and try values such as 0.1, 0.01, 0.001, and 0.0001.
- Change the number of hidden layers (e.g., 1, 2, or 3) and the number of neurons per layer (e.g., 16, 32, 64, 128). The baseline uses only 2 and 6 neurons — can you do better?
- Vary the batch size (e.g., 16, 32, 64). Observe how these affect convergence speed and overfitting.


**In your report, include:**
1. A table with at least 9 configurations, their validation and test losses and accuracies.
2. Loss training curve for your best model.

**Make sure** the jupyter notebook that you will attach with the report includes your best model.

## Submission

Submit the report and the jupyter notebook. 
The report should be maximum 5 pages.

#### Report structure:

Introduction: 
- Describe in two paragraphs clustering and classification tasks.

Methods: 
- Describe the hyperparameters of k-means and neural networks.

Results: 
- Discuss the challenges of k-means by answering tasks 1-4.
- Evaluate the effects of the hyperparameters of neural networks by answering tasks 5-7

Conclusions:
- Summarize key points of your findings.
